In [1]:
import os
import cv2
import torch
import numpy as np
import torch.nn as nn
import torchvision.models as models
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from tqdm import tqdm

# --- CONFIGURATION ---
MODEL_WEIGHTS_PATH = "/kaggle/input/datasets/nithishkannaa/nonutdovemberlovember/best_dual_stream_model.pth"
HF_REPO_URL = "https://huggingface.co/datasets/Nithish0p/dfdc_processed_faces"
DATA_DIR = "/tmp/dfdc_faces"

BATCH_SIZE = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. DOWNLOAD DFDC DATA (Bypassing API limits) ---
if not os.path.exists(DATA_DIR):
    print("Pulling DFDC dataset via native Git stream... (Takes 2-5 mins)")
    os.system(f"git clone {HF_REPO_URL} {DATA_DIR}")
else:
    print("DFDC Dataset already exists in /tmp.")

# --- 2. ARCHITECTURE ---
class SRMConv2d(nn.Module):
    def __init__(self):
        super().__init__()
        srm_kernel = torch.tensor([
            [[-1,  2, -1],
             [ 2, -4,  2],
             [-1,  2, -1]]
        ], dtype=torch.float32).unsqueeze(1) / 4.0
        self.weight = nn.Parameter(srm_kernel.repeat(3, 1, 1, 1), requires_grad=False)
        
    def forward(self, x):
        return nn.functional.conv2d(x, self.weight, stride=1, padding=1, groups=3)

class DualStreamDeepfakeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.rgb_stream = models.efficientnet_b4(weights=None)
        rgb_feature_dim = self.rgb_stream.classifier[1].in_features
        self.rgb_stream.classifier = nn.Identity()
        
        self.srm_filter = SRMConv2d()
        self.freq_stream = models.resnet18(weights=None)
        self.freq_stream.fc = nn.Identity()
        freq_feature_dim = 512
        
        self.fusion = nn.Sequential(
            nn.Linear(rgb_feature_dim + freq_feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 1)
        )
        
    def forward(self, x):
        spatial_features = self.rgb_stream(x)
        noise_x = self.srm_filter(x)
        freq_features = self.freq_stream(noise_x)
        combined = torch.cat((spatial_features, freq_features), dim=1)
        return self.fusion(combined)

# --- 3. LOAD & LOCK WEIGHTS ---
print("Initializing model architecture...")
model = DualStreamDeepfakeDetector()
print(f"Loading trained weights from: {MODEL_WEIGHTS_PATH}")
model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=device))
model.to(device)
model.eval()

# --- 4. DFDC DATA LOADER (Validation Split Only) ---
class DFDCFaceDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        for batch_dir in self.root_dir.glob("batch_*"):
            for label_str, label_idx in {"real": 0, "fake": 1}.items():
                class_dir = batch_dir / label_str
                if class_dir.exists():
                    for img_path in class_dir.glob("*.jpg"):
                        self.image_paths.append(str(img_path))
                        self.labels.append(label_idx)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.transform:
            image = self.transform(image=image)['image']
            
        return image, label

# Standard evaluation transform (No augmentations for testing)
eval_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

full_dataset = DFDCFaceDataset(root_dir=DATA_DIR, transform=eval_transform)

# Extract the exact 20% validation size using a fixed generator seed
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
generator = torch.Generator().manual_seed(42) # Seed ensures consistent splits
_, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size], generator=generator)

print(f"DFDC Validation Pool Ready: {len(val_dataset)} total images loaded.")
dfdc_val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# --- 5. INFERENCE LOOP ---
print("\nStarting Evaluation on DFDC Validation Split...")
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(dfdc_val_loader, desc="Running Inference"):
        images = images.to(device)
        outputs = model(images).squeeze(1)
        probs = torch.sigmoid(outputs).cpu().numpy()
        
        all_preds.extend(probs)
        all_labels.extend(labels.numpy())

# --- 6. CALCULATE METRICS ---
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
binary_preds = (all_preds > 0.5).astype(int)

accuracy = accuracy_score(all_labels, binary_preds)
auc_roc = roc_auc_score(all_labels, all_preds)

print("\n" + "="*40)
print("FINAL DFDC (INTRA-DATASET) BENCHMARK RESULTS")
print("="*40)
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"AUC-ROC:  {auc_roc:.4f}")
print("="*40)
print("\nDetailed Classification Report:")
print(classification_report(all_labels, binary_preds, target_names=["Real (0)", "Fake (1)"]))

Pulling DFDC dataset via native Git stream... (Takes 2-5 mins)


Cloning into '/tmp/dfdc_faces'...
Updating files: 100% (198997/198997), done.
Filtering content: 100% (198996/198996), 2.01 GiB | 533.00 KiB/s, done.


Initializing model architecture...
Loading trained weights from: /kaggle/input/datasets/nithishkannaa/nonutdovemberlovember/best_dual_stream_model.pth
DFDC Validation Pool Ready: 39800 total images loaded.

Starting Evaluation on DFDC Validation Split...


Running Inference: 100%|██████████| 622/622 [02:54<00:00,  3.57it/s]


FINAL DFDC (INTRA-DATASET) BENCHMARK RESULTS
Accuracy: 98.60%
AUC-ROC:  0.9970

Detailed Classification Report:
              precision    recall  f1-score   support

    Real (0)       0.97      0.91      0.94      4922
    Fake (1)       0.99      1.00      0.99     34878

    accuracy                           0.99     39800
   macro avg       0.98      0.95      0.97     39800
weighted avg       0.99      0.99      0.99     39800

